# Lasso

## Импорт библиотек, загрузка датасетов

In [8]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error,  mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Загрузка данных
train = pd.read_csv('data/train_preprocessed.csv')
test = pd.read_csv('data/test_preprocessed.csv')

## Построение модели

In [9]:
X = train.drop('price_per_sqm', axis=1)
y = np.log1p(train['price_per_sqm'])

X_sub = test.drop('id', axis=1)
test_ids = test['id']

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
numeric_cols = ['area', 'year', 'floor', 'total_floors']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols)
], remainder='passthrough')

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Lasso(max_iter=10000))
])

alphas = np.logspace(-2, 3, 20)
param_grid = {'model__alpha': alphas}

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

In [12]:
# fit
grid_search.fit(X_train, y_train_log)
best_alpha = grid_search.best_params_['model__alpha']
print("Best alpha:", best_alpha)
best_model = grid_search.best_estimator_
val_preds_log = best_model.predict(X_test)

val_preds = np.expm1(val_preds_log)
y_test = np.expm1(y_test_log)

Best alpha: 0.01


In [13]:
print("R2:\t %.4f" % r2_score(y_test, val_preds))
print("MAPE:\t %.4f" % mean_absolute_percentage_error(y_test, val_preds))
print('-----------------')
print("MSE:\t %.4f" % mean_squared_error(y_test, val_preds))
print("RMSE:\t %.4f" % np.sqrt(mean_squared_error(y_test, val_preds)))
print("MAE:\t %.4f" % mean_absolute_error(y_test, val_preds))

R2:	 0.0057
MAPE:	 0.1551
-----------------
MSE:	 7291556.0667
RMSE:	 2700.2881
MAE:	 293.2604


## Проверка модели на submission.csv

In [17]:
sub_preds_log = best_model.predict(X_sub)
sub_preds = np.expm1(sub_preds_log)

sub_usd_price = sub_preds * X_sub['area']
submission = pd.DataFrame({'id': test_ids,'usd_price': sub_usd_price})
submission.to_csv('data/Lasso.csv', index=False)